In [1]:
import pandas as pd
import numpy as np
from librosa.util import find_files
import librosa
import os
import pickle
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
import torch.nn.functional as F
from dataset import ASVspoof2019
from evaluate_tDCF_asvspoof19 import compute_eer_and_tdcf
from collections import defaultdict
from tqdm import tqdm
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import eval_metrics as em
from evaluate_tDCF_asvspoof19 import compute_eer_and_tdcf

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [3]:
def test_model(feat_model_path, loss_model_path, part, add_loss):
    dirname = os.path.dirname
    basename = os.path.splitext(os.path.basename(feat_model_path))[0]
    if "checkpoint" in dirname(feat_model_path):
        dir_path = dirname(dirname(feat_model_path))
    else:
        dir_path = dirname(feat_model_path)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = torch.load(feat_model_path)
    # model = torch.nn.DataParallel(model, list(range(torch.cuda.device_count())))  # for multiple GPUs
    loss_model = torch.load(loss_model_path) if add_loss is not None else None
    test_set = ASVspoof2019("LA", "/dataNVME/neil/ASVspoof2019LAFeatures/",
                            "/data/neil/DS_10283_3336/LA/ASVspoof2019_LA_cm_protocols/", part,
                            "LFCC", feat_len=750, padding="repeat")
    testDataLoader = DataLoader(test_set, batch_size=16, shuffle=False, num_workers=0)
    model.eval()
    score_loader, idx_loader = [], []

    with open(os.path.join(dir_path, 'spl_cm_score.txt'), 'w') as cm_score_file:
        for i, (lfcc, audio_fn, tags, labels) in enumerate(tqdm(testDataLoader)):
            lfcc = lfcc.unsqueeze(1).float().to(device)
            # print(lfcc.shape)
            tags = tags.to(device)
            labels = labels.to(device)

            feats, lfcc_outputs = model(lfcc)

            score = F.softmax(lfcc_outputs)[:, 0]
            # print(score)

            if add_loss == "ocsoftmax":
                ang_isoloss, score = loss_model(feats, labels)
            elif add_loss == "amsoftmax":
                outputs, moutputs = loss_model(feats, labels)
                score = F.softmax(outputs, dim=1)[:, 0]
            else: pass

            for j in range(labels.size(0)):
                cm_score_file.write(
                    '%s A%02d %s %s\n' % (audio_fn[j], tags[j].data,
                                          "spoof" if labels[j].data.cpu().numpy() else "bonafide",
                                          score[j].item()))

            score_loader.append(score.detach().cpu())
            idx_loader.append(labels.detach().cpu())

    scores = torch.cat(score_loader, 0).data.cpu().numpy()
    labels = torch.cat(idx_loader, 0).data.cpu().numpy()
    eer = em.compute_eer(scores[labels == 0], scores[labels == 1])[0]
    other_eer = em.compute_eer(-scores[labels == 0], -scores[labels == 1])[0]
    eer = min(eer, other_eer)

    return eer


def test_individual_attacks(cm_score_file):
    # Load CM scores
    cm_data = np.genfromtxt(cm_score_file, dtype=str)
    cm_sources = cm_data[:, 1]
    cm_keys = cm_data[:, 2]
    cm_scores = cm_data[:, 3].astype(np.float)

    other_cm_scores = -cm_scores

    eer_cm_lst, min_tDCF_lst = [], []
    for attack_idx in range(7, 20):
        # Extract target, nontarget, and spoof scores from the ASV scores

        # Extract bona fide (real human) and spoof scores from the CM scores
        bona_cm = cm_scores[cm_keys == 'bonafide']
        spoof_cm = cm_scores[cm_sources == 'A%02d' % attack_idx]

        # EERs of the standalone systems and fix ASV operating point to EER threshold
        eer_cm = em.compute_eer(bona_cm, spoof_cm)[0]

        other_eer_cm = em.compute_eer(other_cm_scores[cm_keys == 'bonafide'], other_cm_scores[cm_sources == 'A%02d' % attack_idx])[0]

        eer_cm_lst.append(min(eer_cm, other_eer_cm))

    return eer_cm_lst

In [4]:
def test(model_dir, add_loss):
    if "antiRes" in model_dir:
        model_path = os.path.join(model_dir, "anti-spoofing_lfcc_model.pt")
    else:
        model_path = os.path.join(model_dir, "anti-spoofing_cqcc_model.pt")
    loss_model_path = os.path.join(model_dir, "anti-spoofing_loss_model.pt")
    val_eer = test_model(model_path, loss_model_path, "dev", add_loss)
    eer = test_model(model_path, loss_model_path, "eval", add_loss)
    eer_cm_lst = test_individual_attacks(os.path.join(model_dir, 'spl_cm_score.txt'))
    return val_eer, eer, eer_cm_lst

In [5]:
val_eer_oc, eer_oc, eer_cm_lst_oc = test("/data/neil/antiRes/models1028/ocsoftmax", "ocsoftmax")

/home/neil/anaconda3/envs/analyse/lib/python3.6/site-packages/torch/serialization.py:454: SourceChangeWarning: source code of class 'resnet.ResNet' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  warnings.warn(msg, SourceChangeWarning)
100%|██████████| 4453/4453 [11:04<00:00,  6.70it/s]


In [14]:
eer_oc

0.021779869201821823

In [11]:
val_eer_am, eer_am, eer_cm_lst_am = test("/data/neil/antiRes/models1028/amsoftmax", "amsoftmax")

100%|██████████| 4453/4453 [08:15<00:00,  8.98it/s]


In [15]:
eer_am

0.03164996534760693

In [34]:
val_eer_ce, eer_ce, eer_cm_lst_ce = test("/data/neil/antiRes/models1028/softmax2", None)

100%|██████████| 4453/4453 [08:04<00:00,  9.19it/s]


In [25]:
def stat_anal(eera, eerb, nbona, nspoof):
    nomi = 2 * np.abs(eera - eerb)
    deno = np.sqrt((eera * (1-eera) + eerb * (1-eerb)) * (nbona + nspoof) / nbona / nspoof)
    return nomi / deno

In [32]:
for i in range(13):
    print("A%02d" % (i+7), stat_anal(eer_cm_lst_oc[i], eer_cm_lst_am[i], 7355, 4914))
print("Overall", stat_anal(eer_oc, eer_am, 7355, 4914*13))

A07 1.8268545050048441
A08 2.5362201433436637
A09 3.0641189116809318
A10 4.204659936205309
A11 2.7986586871641506
A12 0.7038984495129539
A13 0.7389718821836467
A14 1.8573715530362946
A15 5.353397157257477
A16 2.1463946813750687
A17 10.326569997728898
A18 16.364380734800044
A19 0.1407599564009545
Overall 7.03348721234126


In [35]:
for i in range(13):
    print("A%02d" % (i+7), stat_anal(eer_cm_lst_oc[i], eer_cm_lst_ce[i], 7355, 4914))
print("Overall", stat_anal(eer_oc, eer_ce, 7355, 4914*13))

A07 3.800957533030602
A08 0.7835234151586435
A09 2.7986586871641506
A10 0.2919224290483776
A11 3.517582835542615
A12 0.39008356819012735
A13 5.376259339659859
A14 20.014100984487953
A15 17.143913886793914
A16 1.438747119744761
A17 30.214967052024953
A18 7.196299152534899
A19 3.460749628865748
Overall 17.765306030219914


In [37]:
for i in range(13):
    print("A%02d" % (i+7), stat_anal(eer_cm_lst_ce[i], eer_cm_lst_am[i], 7355, 4914))
print("Overall", stat_anal(eer_ce, eer_am, 7355, 4914*13))

A07 2.0711575857433084
A08 1.797348975928013
A09 0.3656231152345899
A10 4.488113933076317
A11 5.730387422600706
A12 1.0930037638790586
A13 4.713459431710971
A14 21.24864208816508
A15 21.327711143866388
A16 3.5364343486107925
A17 20.000423152940918
A18 21.262102000991685
A19 3.598931281456541
Overall 10.96120929446067
